<a href="https://colab.research.google.com/github/jakkrol/machine-learning/blob/main/DeepfakeDetector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

os.environ['KAGGLE_USERNAME'] = ""
os.environ['KAGGLE_KEY'] = ""


!mkdir -p ~/.kaggle
import json
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': os.environ['KAGGLE_USERNAME'], 'key': os.environ['KAGGLE_KEY']}, f)

!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle zostało skonfigurowane bezpośrednio z kodu!")

Kaggle zostało skonfigurowane bezpośrednio z kodu!


In [ ]:
import os
import zipfile

!kaggle datasets download -d tristanzhang32/ai-generated-images-vs-real-images

print("Rozpakowywanie paczki cifake...")
with zipfile.ZipFile("ai-generated-images-vs-real-images.zip", 'r') as zip_ref:
    zip_ref.extractall("learning_data")

print("Rozpakowywanie zakończone pomyślnie!")

base_dir = 'learning_data'
if os.path.exists(base_dir):
    print("\nStruktura katalogów w środku:")
    print(os.listdir(base_dir))

Dataset URL: https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images
License(s): other
100% 105M/105M [00:00<00:00, 128MB/s] 

Rozpakowywanie paczki cifake...
Rozpakowywanie zakończone pomyślnie!

Struktura katalogów w środku:
['train', 'test']


In [ ]:
import os
import cv2
import glob
from tqdm import tqdm

# --- 2. DEFINICJA FUNKCJI KAFELKUJĄCEJ ---
def slice_dataset(source_dir, target_dir, tile_size=224, stride=224):
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
    image_paths = []
    for ext in extensions:
        image_paths.extend(glob.glob(os.path.join(source_dir, '**', ext), recursive=True))

    print(f"📦 Znaleziono {len(image_paths)} dużych zdjęć w {source_dir}. Rozpoczynam cięcie...")

    for path in tqdm(image_paths):
        img = cv2.imread(path)
        if img is None:
            continue

        h, w, _ = img.shape
        rel_path = os.path.relpath(path, source_dir)
        sub_dir = os.path.dirname(rel_path)
        base_name = os.path.splitext(os.path.basename(path))[0]

        out_folder = os.path.join(target_dir, sub_dir)
        os.makedirs(out_folder, exist_ok=True)

        tile_count = 0
        for y in range(0, h - tile_size + 1, stride):
            for x in range(0, w - tile_size + 1, stride):
                tile = img[y:y+tile_size, x:x+tile_size]
                if tile.shape[0] != tile_size or tile.shape[1] != tile_size:
                    continue

                tile_name = f"{base_name}_tile_{tile_count}.jpg"
                cv2.imwrite(os.path.join(out_folder, tile_name), tile)
                tile_count += 1

# --- 3. RESTRUKTURYZACJA: URUCHOMIENIE CIĘCIA (NOWOŚĆ!) ---
# Tniemy dane z 'learning_data' i zapisujemy w skali 1:1 do 'dataset_pociety'
print("\n--- Przygotowanie kafelków 1:1 ---")
slice_dataset('learning_data/train', 'dataset_pociety/train', tile_size=224, stride=112)
slice_dataset('learning_data/test', 'dataset_pociety/test', tile_size=224, stride=112)


In [ ]:
# import os

# # Definiujemy ścieżkę do rozpakowanego folderu
# dataset_path = 'deepfake_faces_data/Final Dataset'

# if os.path.exists(dataset_path):
#     print("Zawartość folderu 'Final Dataset':")
#     subfolders = os.listdir(dataset_path)
#     print(subfolders)

#     # Sprawdźmy głębiej, co jest w środku pierwszego z brzegu podfolderu
#     for folder in subfolders:
#         full_sub_path = os.path.join(dataset_path, folder)
#         if os.path.isdir(full_sub_path):
#             print(f"\nLiczba plików w folderze '{folder}': {len(os.listdir(full_sub_path))}")
#             # Pokazuje 3 przykładowe nazwy plików
#             print(f"Przykładowe pliki: {os.listdir(full_sub_path)[:3]}")
# else:
#     print("Coś poszło nie tak, nie znaleziono folderu 'Final Dataset'.")

In [ ]:
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory

#base_dir = 'deepfake_faces_data/Final Dataset'
train_dir = 'dataset_pociety/train'
test_dir = 'dataset_pociety/test'
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

print("\n--- ładowanie zbioru treningowego ---")
train_ds = image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("\n--- ładowanie zbioru testowego ---")
val_ds = image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("\n--- Weryfikacja matematyczna klas przez Keras ---")
for i, name in enumerate(class_names):
    print(f"Wartość {i}.0 na wyjściu modelu (Sigmoid) oznacza klasę: {name}")


--- ładowanie zbioru treningowego ---
Found 100000 files belonging to 2 classes.

--- ładowanie zbioru testowego ---
Found 20000 files belonging to 2 classes.

Wykryte klasy: ['FAKE', 'REAL'] (Gdzie: FAKE = Prawdziwe, REAL = Deepfake)


In [ ]:
from tensorflow.keras import layers, models

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

base_model = tf.keras.applications.MobileNetV2(
    input_shape = (224, 224, 3),
    include_top = False,
    weights = "imagenet"
)

base_model.trainable = False



model = models.Sequential([
    layers.RandomFlip("horizontal", input_shape=(224, 224, 3)),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',                                     # Najpopularniejszy, stabilny algorytm uczący
    loss='binary_crossentropy',                           # Standardowa funkcja strat dla problemów Tak/Nie
    metrics=['accuracy']                                  # Chcemy na bieżąco widzieć skuteczność w %
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_brightness               │ (None, 224, 224, 3)    │             0 │
│ (RandomBrightness)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
print("--start training--")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5
)

--start training--
Epoch 1/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 132s 36ms/step - accuracy: 0.8301 - loss: 0.3791 - val_accuracy: 0.8505 - val_loss: 0.3385
Epoch 2/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 109s 35ms/step - accuracy: 0.8476 - loss: 0.3470 - val_accuracy: 0.8666 - val_loss: 0.3114
Epoch 3/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 105s 34ms/step - accuracy: 0.8474 - loss: 0.3490 - val_accuracy: 0.8673 - val_loss: 0.3101
Epoch 4/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 106s 34ms/step - accuracy: 0.8484 - loss: 0.3477 - val_accuracy: 0.8595 - val_loss: 0.3221
Epoch 5/5
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 142s 34ms/step - accuracy: 0.8480 - loss: 0.3482 - val_accuracy: 0.8619 - val_loss: 0.3175


In [ ]:
print("--Dostrajanie modelu--")

unfreezeModel = model.layers[4]
unfreezeModel.trainable = True

for layer in unfreezeModel.layers[:-30]:
  layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

history_f = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=7
)


--Dostrajanie modelu--


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_brightness               │ (None, 224, 224, 3)    │             0 │
│ (RandomBrightness)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_contrast                 │ (None, 224, 224, 3)    │             0 │
│ (RandomContrast)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,527,681 (5.83 MB)

 Non-trainable params: 731,584 (2.79 MB)

Epoch 1/7
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 155s 44ms/step - accuracy: 0.9145 - loss: 0.2142 - val_accuracy: 0.8984 - val_loss: 0.2699
Epoch 2/7
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 189s 43ms/step - accuracy: 0.9508 - loss: 0.1260 - val_accuracy: 0.9186 - val_loss: 0.2169
Epoch 3/7
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 140s 42ms/step - accuracy: 0.9639 - loss: 0.0932 - val_accuracy: 0.9532 - val_loss: 0.1227
Epoch 4/7
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 133s 42ms/step - accuracy: 0.9711 - loss: 0.0747 - val_accuracy: 0.9593 - val_loss: 0.1098
Epoch 5/7
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 133s 42ms/step - accuracy: 0.9776 - loss: 0.0602 - val_accuracy: 0.9641 - val_loss: 0.1010
Epoch 6/7
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 142s 42ms/step - accuracy: 0.9815 - loss: 0.0497 - val_accuracy: 0.9461 - val_loss: 0.1670
Epoch 7/7
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 136s 43ms/step - accuracy: 0.9854 - loss: 0.0403 - val_accuracy: 0.9662 - val_loss: 0.0983


In [ ]:
model.save('deepfake_v2.keras')

from google.colab import files
files.download('deepfake_v2.keras')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!pip install -q gradio
import gradio as gr
import tensorflow as tf
import numpy as np
import cv2

# 1. Ładowanie modelu
model = tf.keras.models.load_model(
    'deepfake_v2.keras',
    custom_objects={'preprocess_input': tf.keras.applications.mobilenet_v2.preprocess_input}
)

def predict_deepfake_scan(img):
    if img is None:
        return "Proszę wrzucić zdjęcie."

    # Zapewniamy standardowy format obrazu (0-255) do operacji na OpenCV
    img = np.array(img, dtype=np.uint8)

    # --- 1. Skalowanie wstępne całego obrazu (Przed konwersją do float!) ---
    target_mid_size = 600
    h_orig, w_orig, _ = img.shape

    if h_orig > target_mid_size or w_orig > target_mid_size:
        if h_orig < w_orig:
            new_h = target_mid_size
            new_w = int(w_orig * (target_mid_size / h_orig))
        else:
            new_w = target_mid_size
            new_h = int(h_orig * (target_mid_size / w_orig))
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)

    # Teraz bezpiecznie przechodzimy na float32 przed preprocessingiem Keras
    img_float = img.astype(np.float32)

    # --- 2. GLOBALNY PREPROCESSING ---
    img_ready = np.expand_dims(img_float, axis=0)
    img_ready = tf.keras.applications.mobilenet_v2.preprocess_input(img_ready)

    # Bezpieczne wyciągnięcie czystej macierzy numpy z tensora
    img_ready = img_ready[0].numpy() if hasattr(img_ready, 'numpy') else img_ready[0]

    h, w, _ = img_ready.shape
    tile_size = 224
    stride = 112

    lowest_real_score = 1.0

    # Przesuwamy się po wysokości (Y) i szerokości (X)
    for y in range(0, h - tile_size + 1, stride):
        for x in range(0, w - tile_size + 1, stride):

            # Wycinamy kafelek z znormalizowanego globalnie obrazu
            tile = img_ready[y:y+tile_size, x:x+tile_size]

            if tile.shape[0] != tile_size or tile.shape[1] != tile_size:
                continue

            # Przygotowanie wymiaru wejściowego dla sieci (batch size = 1)
            tile_array = np.expand_dims(tile, axis=0)

            # Predykcja
            prediction = model.predict(tile_array, verbose=0)
            score = prediction[0][0]

            if score < lowest_real_score:
                lowest_real_score = score

            if lowest_real_score < 0.05:
                break

    # Ostateczny warunek interpretacji (0 = FAKE, 1 = REAL)
    if lowest_real_score < 0.5:
        confidence = (1 - lowest_real_score) * 100
        return f"🚨 WYKRYTO GENERACJĘ AI! (Pewność sztuczności: {confidence:.2f}%)"
    else:
        confidence = lowest_real_score * 100
        return f"✅ PRAWDZIWE ZDJĘCIE (Pewność: {confidence:.2f}%)"

# 3. Interfejs graficzny
interface = gr.Interface(
    fn=predict_deepfake_scan,
    inputs=gr.Image(),
    outputs="text",
    title="Wykrywacz Deepfake z globalną normalizacją",
    description="Skanowanie kafelkowe z zachowaniem globalnych statystyk obrazu RGB."
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://79442bfd59fd005fb1.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
